In [3]:
class CharTokenizer:
    """ Simple Tokenizer which tokenizes at character level and converts chars to ASCII codes"""

    def encode(self, text):
        """Char to ASCII"""
        return [ord(c) for c in text]

    def decode(self, tokens):
        ## ASCII to char
        return "".join(chr(t) for t in tokens)

CharTokenizer().encode("Hello World")

[72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100]

### BPE Tokenizer from scratch

In [5]:
from collections import Counter

class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {}

    def _get_pairs(self, tokens):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i + 1])] += 1
        return pairs

    def _merge_pair(self, tokens, pair, new_token):
        merged = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                merged.append(new_token)
                i += 2
            else:
                merged.append(tokens[i])
                i += 1
        return merged

    def train(self, text, num_merges):
        tokens = list(text.encode("utf-8"))
        self.vocab = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):
            pairs = self._get_pairs(tokens)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            new_token = 256 + i
            tokens = self._merge_pair(tokens, best_pair, new_token)
            self.merges[best_pair] = new_token
            self.vocab[new_token] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

        return self

    def encode(self, text):
        tokens = list(text.encode("utf-8"))
        for pair, new_token in self.merges.items():
            tokens = self._merge_pair(tokens, pair, new_token)
        return tokens

    def decode(self, tokens):
        byte_sequence = b"".join(self.vocab[t] for t in tokens)
        return byte_sequence.decode("utf-8", errors="replace")


lib = BPETokenizer()
lib.train("hello I am looking for a word text",5)

In [6]:
lib.vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [13]:
[(chr(i[0])+f":{i[0]}" , chr(i[1]) +f":{i[1]}") for i in lib.merges]

[('l:108', 'o:111'),
 (' :32', 'a:97'),
 ('o:111', 'r:114'),
 ('h:104', 'e:101'),
 ('ă:259', 'l:108')]

In [23]:
lib.encode("Test Look")

[84, 101, 115, 116, 32, 76, 111, 111, 107]

The training loop is the core of BPE: count pairs, merge the winner, repeat. Each merge reduces the total token count. After num_merges rounds, the vocabulary grows from 256 (base bytes) to 256 + num_merges.

Encoding applies merges in the exact order they were learned. This matters. If merge 1 created "th" and merge 5 created "the", encoding must apply merge 1 first so that "the" can form from "th" + "e" in merge 5.

Decoding is the inverse: look up each token ID in the vocabulary, concatenate the bytes, decode to UTF-8.

In [24]:
lib.decode([84, 101, 115, 116, 32, 76,])

'Test L'

In [28]:
### Compression Ratio

len(lib.encode("Hello World"))/len("Hello World")

## Less is better 


0.8181818181818182

In [30]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

texts = [
    "The cat sat on the mat.",
    "unhappiness",
    "Hello, world!",
    "def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)",
    "Geschwindigkeitsbegrenzung",
]

for text in texts:
    our_tokens = lib.encode(text)
    tiktoken_tokens = enc.encode(text)
    tiktoken_pieces = [enc.decode([t]) for t in tiktoken_tokens]
    print(f"'{text}'")
    print(f"  Our BPE:   {len(our_tokens)} tokens")
    print(f"  tiktoken:  {len(tiktoken_tokens)} tokens -> {tiktoken_pieces}")

'The cat sat on the mat.'
  Our BPE:   21 tokens
  tiktoken:  7 tokens -> ['The', ' cat', ' sat', ' on', ' the', ' mat', '.']
'unhappiness'
  Our BPE:   11 tokens
  tiktoken:  3 tokens -> ['un', 'h', 'appiness']
'Hello, world!'
  Our BPE:   11 tokens
  tiktoken:  4 tokens -> ['Hello', ',', ' world', '!']
'def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)'
  Our BPE:   72 tokens
  tiktoken:  23 tokens -> ['def', ' fibonacci', '(n', '):', ' return', ' n', ' if', ' n', ' <', ' ', '2', ' else', ' fibonacci', '(n', '-', '1', ')', ' +', ' fibonacci', '(n', '-', '2', ')']
'Geschwindigkeitsbegrenzung'
  Our BPE:   26 tokens
  tiktoken:  9 tokens -> ['G', 'esch', 'wind', 'ig', 'ke', 'its', 'beg', 'ren', 'zung']


## Analyse Vocab

In [31]:
def analyze_vocabulary(tokenizer, test_texts):
    total_tokens = 0
    total_chars = 0
    token_usage = Counter()

    for text in test_texts:
        encoded = tokenizer.encode(text)
        total_tokens += len(encoded)
        total_chars += len(text)
        for t in encoded:
            token_usage[t] += 1

    print(f"Vocabulary size: {len(tokenizer.vocab)}")
    print(f"Total tokens across all texts: {total_tokens}")
    print(f"Total characters: {total_chars}")
    print(f"Avg tokens per character: {total_tokens / total_chars:.2f}")

    print(f"\nMost used tokens:")
    for token_id, count in token_usage.most_common(10):
        token_bytes = tokenizer.vocab[token_id]
        display = token_bytes.decode("utf-8", errors="replace")
        print(f"  Token {token_id:4d}: '{display}' (used {count} times)")

    unused = [t for t in tokenizer.vocab if t not in token_usage]
    print(f"\nUnused tokens: {len(unused)} out of {len(tokenizer.vocab)}")

analyze_vocabulary(lib,texts)


Vocabulary size: 261
Total tokens across all texts: 141
Total characters: 145
Avg tokens per character: 0.97

Most used tokens:
  Token   32: ' ' (used 17 times)
  Token  110: 'n' (used 15 times)
  Token  105: 'i' (used 11 times)
  Token  101: 'e' (used 10 times)
  Token   99: 'c' (used 8 times)
  Token   97: 'a' (used 7 times)
  Token  116: 't' (used 6 times)
  Token  115: 's' (used 6 times)
  Token  102: 'f' (used 5 times)
  Token  111: 'o' (used 4 times)

Unused tokens: 222 out of 261
